# Retail Spain Sales Analysis Pipeline

## Project Overview
This project analyzes retail sales in Spain using PySpark and Databricks.

The pipeline includes:
- Data extraction from CSV
- Data cleaning and transformation
- Year-over-Year (YoY) growth analysis
- Sales participation analysis
- Top regional rankings

## Technologies
- Databricks
- PySpark
- Spark SQL Functions
- Window Functions

In [0]:
# ======================================
# IMPORTS
# ======================================

from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    regexp_replace,
    expr,
    sum,
    substring,
    lag,
    round,
    dense_rank
)


In [0]:
# ======================================
# EXTRACT
# ======================================

df = spark.read.csv(
    "dbfs:/Volumes/workspace/default/raw_data/ventas_ine_comercio_minorista.csv",
    header=True,
    inferSchema=True,
    sep=";",
    encoding="latin1"
)

In [0]:
display(df)

Región Geográfica,Variables,SECCIONES,DIVISIONES,GRUPOS,Periodo,Total
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,50.938.080
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,48.065.949
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,44.948.623
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,7.948.689
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,7.585.206
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,7.314.981
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,5.944.579
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,5.527.319
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,5.418.920
"Balears, Illes",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,10.157.375


In [0]:
df.printSchema()

root
 |-- Región Geográfica: string (nullable = true)
 |-- Variables: string (nullable = true)
 |-- SECCIONES : string (nullable = true)
 |-- DIVISIONES: string (nullable = true)
 |-- GRUPOS: string (nullable = true)
 |-- Periodo: integer (nullable = true)
 |-- Total: string (nullable = true)



In [0]:
df.columns

['Región Geográfica',
 'Variables',
 'SECCIONES ',
 'DIVISIONES',
 'GRUPOS',
 'Periodo',
 'Total']

In [0]:
# ======================================
# DATA CLEANING
# ======================================

df_clean = df.select(
    col("Región Geográfica").alias("comunidad"),
    col("Variables").alias("variable"),
    col("SECCIONES ").alias("seccion"),
    col("DIVISIONES").alias("division"),
    col("GRUPOS").alias("grupo"),
    col("Periodo").alias("periodo"),
    col("Total").alias("total")
)

In [0]:
df_clean.printSchema()
display(df_clean)

root
 |-- comunidad: string (nullable = true)
 |-- variable: string (nullable = true)
 |-- seccion: string (nullable = true)
 |-- division: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- periodo: integer (nullable = true)
 |-- total: string (nullable = true)



comunidad,variable,seccion,division,grupo,periodo,total
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,50.938.080
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,48.065.949
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,44.948.623
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,7.948.689
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,7.585.206
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,7.314.981
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,5.944.579
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,5.527.319
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,5.418.920
"Balears, Illes",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,10.157.375


In [0]:
display(
    df_clean.select("total")
)

total
50.938.080
48.065.949
44.948.623
7.948.689
7.585.206
7.314.981
5.944.579
5.527.319
5.418.920
10.157.375


In [0]:
df_clean = df_clean.withColumn(
    "total",
    regexp_replace(col("total"), r"\.", "")
)

df_clean = df_clean.withColumn(
    "total",
    expr("try_cast(total AS BIGINT)")
)

In [0]:
df_clean.printSchema()
display(df_clean)

root
 |-- comunidad: string (nullable = true)
 |-- variable: string (nullable = true)
 |-- seccion: string (nullable = true)
 |-- division: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- periodo: integer (nullable = true)
 |-- total: long (nullable = true)



comunidad,variable,seccion,division,grupo,periodo,total
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,50938080
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,48065949
Andalucía,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,44948623
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,7948689
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,7585206
Aragón,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,7314981
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,5944579
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,5527319
"Asturias, Principado de",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,5418920
"Balears, Illes",Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,10157375


In [0]:
df_clean.filter(col("total").isNull()).count()

6

In [0]:
df_clean.select("total")

DataFrame[total: bigint]

In [0]:
display(
    df_clean.select("total")
)

total
50938080
48065949
44948623
7948689
7585206
7314981
5944579
5527319
5418920
10157375


In [0]:
display(
    df_clean
        .filter(col("total")
        .isNull()))

comunidad,variable,seccion,division,grupo,periodo,total
Ceuta,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,null
Ceuta,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,null
Ceuta,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,null
Melilla,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2024,null
Melilla,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2023,null
Melilla,Cifra de negocios,G TOTAL SECTOR COMERCIO,"47 Comercio al por menor, excepto de vehículos de motor y motocicletas",null,2022,null


In [0]:
df_clean.count()

57

In [0]:
display(
    df_clean
        .select("comunidad")
        .distinct()
        .orderBy("comunidad")
)

comunidad
Andalucía
Aragón
"Asturias, Principado de"
"Balears, Illes"
Canarias
Cantabria
Castilla - La Mancha
Castilla y León
Cataluña
Ceuta


In [0]:
# ======================================
# KPI 1: Ventas por comunidad y año
# ======================================

ventas_comunidad = (
    df_clean
        .withColumn(
            "anio",
            substring(col("periodo").cast("string"), 1, 4)
        )
        .groupBy("comunidad", "anio")
        .agg(sum("total").alias("ventas"))
        .orderBy("comunidad", "anio")
)

In [0]:
display(ventas_comunidad)

comunidad,anio,ventas
Andalucía,2022,44948623
Andalucía,2023,48065949
Andalucía,2024,50938080
Aragón,2022,7314981
Aragón,2023,7585206
Aragón,2024,7948689
"Asturias, Principado de",2022,5418920
"Asturias, Principado de",2023,5527319
"Asturias, Principado de",2024,5944579
"Balears, Illes",2022,9770928


In [0]:
# =========================================
# KPI 2: Crecimiento YoY (%) por comunidad
# =========================================

window_spec = Window.partitionBy(
    "comunidad"
).orderBy(
    "anio"
)

crecimiento_yoy = (
    ventas_comunidad
        .withColumn(
            "ventas_anio_anterior",
            lag("ventas").over(window_spec)
        )
        .withColumn(
            "crecimiento_yoy",
            round(
                (
                    (col("ventas") - col("ventas_anio_anterior"))
                    / col("ventas_anio_anterior")
                ) * 100,
                2
            )
        )
)

display(crecimiento_yoy)

comunidad,anio,ventas,ventas_anio_anterior,crecimiento_yoy
Andalucía,2022,44948623,null,null
Andalucía,2023,48065949,44948623,6.94
Andalucía,2024,50938080,48065949,5.98
Aragón,2022,7314981,null,null
Aragón,2023,7585206,7314981,3.69
Aragón,2024,7948689,7585206,4.79
"Asturias, Principado de",2022,5418920,null,null
"Asturias, Principado de",2023,5527319,5418920,2.0
"Asturias, Principado de",2024,5944579,5527319,7.55
"Balears, Illes",2022,9770928,null,null


In [0]:
# ======================================
# KPI 3: Participación de ventas (%)
# ======================================

window_anio = Window.partitionBy("anio")

participacion_ventas = (
    ventas_comunidad
        .withColumn(
            "ventas_total_anio",
            sum("ventas").over(window_anio)
        )
        .withColumn(
            "participacion_pct",
            round(
                (col("ventas") / col("ventas_total_anio")) * 100,
                2
            )
        )
        .orderBy("anio", "participacion_pct")
)

display(participacion_ventas)

comunidad,anio,ventas,ventas_total_anio,participacion_pct
Ceuta,2022,null,285973615,null
Melilla,2022,null,285973615,null
"Rioja, La",2022,1778528,285973615,0.62
Cantabria,2022,3748793,285973615,1.31
"Navarra, Comunidad Foral de",2022,4336154,285973615,1.52
Extremadura,2022,5234856,285973615,1.83
"Asturias, Principado de",2022,5418920,285973615,1.89
Aragón,2022,7314981,285973615,2.56
"Murcia, Región de",2022,8870129,285973615,3.1
"Balears, Illes",2022,9770928,285973615,3.42


In [0]:
display(
    ventas_comunidad
        .groupBy("anio")
        .agg(
            sum("ventas").alias("ventas_totales_espana")
        )
        .orderBy("anio")
)

anio,ventas_totales_espana
2022,285973615
2023,300841568
2024,319845386


In [0]:
# ======================================
# KPI 4: Top 5 comunidades con mayor participación
# ======================================

window_rankinng = Window.partitionBy(
    "anio"
).orderBy(
    col("participacion_pct").desc()
)

top_5_participacion = (
    participacion_ventas
        .withColumn(
            "ranking",
            dense_rank().over(window_rankinng)
        )
        .filter(col("ranking") <= 5)
        .orderBy("anio", "ranking"
        )
)
display(top_5_participacion)

comunidad,anio,ventas,ventas_total_anio,participacion_pct,ranking
Cataluña,2022,50209103,285973615,17.56,1
"Madrid, Comunidad de",2022,46892335,285973615,16.4,2
Andalucía,2022,44948623,285973615,15.72,3
Comunitat Valenciana,2022,29645367,285973615,10.37,4
Galicia,2022,17001066,285973615,5.94,5
Cataluña,2023,51463386,300841568,17.11,1
"Madrid, Comunidad de",2023,50957461,300841568,16.94,2
Andalucía,2023,48065949,300841568,15.98,3
Comunitat Valenciana,2023,31801349,300841568,10.57,4
Galicia,2023,17539483,300841568,5.83,5


In [0]:
# ============================================================
# KPI 5: Comunidad con mayor crecimiento Year-over-Year (YoY)
# ============================================================

window_ranking_crecimiento = Window.partitionBy(
    "anio"
).orderBy(
    col("crecimiento_yoy").desc()
)

top_crecimiento = (
    crecimiento_yoy
        .filter(
            col("crecimiento_yoy").isNotNull()
        )
        .withColumn(
            "ranking_crecimiento",
            dense_rank().over(window_ranking_crecimiento)
        )
        .filter(
            col("ranking_crecimiento") == 1
        )
        .orderBy("anio")
)

display(top_crecimiento)

comunidad,anio,ventas,ventas_anio_anterior,crecimiento_yoy,ranking_crecimiento
Cantabria,2023,4234613,3748793,12.96,1
Canarias,2024,17887298,15801205,13.2,1
